In [0]:
!cd ../../.. && source scripts/install_on_dbx.sh

In [0]:
%restart_python

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from site import addsitepackages

addsitepackages(None)

In [0]:
import os
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from project.core.dtypes.dataset import TableSpec
from project.core.dtypes.options import DatasetOptions, PreprocessingOptions
from project.data.data_extractor.example import ExampleDataExtractor
from project.data.datamodules.example import ExampleDataModule

In [0]:
example_datamodule = ExampleDataModule(
    input_table_spec=TableSpec(
        table_identifier="dlh.tmp.demo_processed",
        train_table_sample=80.0,
        eval_table_sample=10.0,
        test_table_sample=10.0,
    ),
    data_extractor=ExampleDataExtractor(),
    dataset_options=DatasetOptions(num_proc=10),
    features_preprocessing_options=PreprocessingOptions(nsamples=100),
)

In [0]:
example_datamodule.prepare_data()
example_datamodule.setup()

In [0]:
next(iter(example_datamodule.train_dataloader())).size()

In [0]:
fig, ax = plt.subplots(1, 3, figsize=(12, 6))
train_target = pd.read_parquet(example_datamodule.train_data.dataset_url, columns=["target"])
eval_target = pd.read_parquet(example_datamodule.eval_data.dataset_url, columns=["target"])
test_target = pd.read_parquet(example_datamodule.test_data.dataset_url, columns=["target"])
sns.histplot(train_target, x="target", bins=2, stat="probability", ax=ax[0])
ax[0].set_title("Train Data")
sns.histplot(eval_target, x="target", bins=2, stat="probability", ax=ax[1])
ax[1].set_title("Eval Data")
sns.histplot(test_target, x="target", bins=2, stat="probability", ax=ax[2])
ax[2].set_title("Test Data")
plt.tight_layout()

In [0]:
spark.sql(
    f"DROP TABLE IF EXISTS hive_metastore.default.{example_datamodule.train_data.dataset_url.split('/')[-1]}"
)
spark.sql(
    f"DROP TABLE IF EXISTS hive_metastore.default.{example_datamodule.eval_data.dataset_url.split('/')[-1]}"
)
spark.sql(
    f"DROP TABLE IF EXISTS hive_metastore.default.{example_datamodule.test_data.dataset_url.split('/')[-1]}"
)